# Text-to-SQL: Bridging the Gap Between Human Language and Databases


Text-to-SQL, also known as Natural Language to SQL (NL2SQL), is a rapidly evolving technology that translates natural, everyday language into Structured Query Language (SQL) commands. This innovative approach empowers users to interact with and retrieve data from databases simply by asking questions in plain English, eliminating the need for specialized knowledge of complex SQL syntax.

At its core, Text-to-SQL acts as an intelligent translator. It leverages the power of artificial intelligence, particularly **Natural Language Processing (NLP)** and sophisticated **AI models**, to understand the user's intent and generate the corresponding SQL query. This process allows individuals without a technical background to explore and analyze data, thereby democratizing data access within an organization.



# TANGENT Zero
Let's take a minute to look at https://bird-bench.github.io/

http://mockaroo.com

## Retrieve data

In [1]:
!git clone https://github.com/AshishJangra27/datasets.git


Cloning into 'datasets'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 359 (delta 28), reused 54 (delta 9), pack-reused 235 (from 1)
Receiving objects: 100% (359/359), 283.76 MiB | 11.42 MiB/s, done.
Resolving deltas: 100% (154/154), done.
Updating files: 100% (233/233), done.


## Setup database

In [2]:
import os
import pandas as pd

base_path = "datasets/Sneaker Sales"

# see what files are there
os.listdir(base_path)


['suppliers.csv',
 'sales_reprentative.csv',
 'README.md',
 'customers.csv',
 'sales.csv',
 'products.csv']

In [3]:
customers_df          = pd.read_csv(os.path.join(base_path, "customers.csv"))
products_df           = pd.read_csv(os.path.join(base_path, "products.csv"))
sales_df              = pd.read_csv(os.path.join(base_path, "sales.csv"))
sales_representative_df = pd.read_csv(os.path.join(base_path, "sales_reprentative.csv"))
suppliers_df          = pd.read_csv(os.path.join(base_path, "suppliers.csv"))

customers_df.head(), products_df.head(), sales_df.head(), sales_representative_df.head(), suppliers_df.head()

(  CustomerID FirstName LastName                   Email      PhoneNumber  \
 0       C001     Rahul   Sharma  rahul.sharma@email.com  +91 98765 43210   
 1       C002    priya     Patel   priya.patel@email.com  +91 87654 32109   
 2       C003      amit    Singh    amit.singh@email.com  +91 76543 21098   
 3       C004      Neha    Gupta    neha.gupta@email.com  +91 65432 10987   
 4       C005    Vikram    Reddy  vikram.reddy@email.com  +91 54321 09876   
 
          City        State  
 0   bangalore    Karnataka  
 1       Delhi        Delhi  
 2      mumbai  Maharashtra  
 3    Chennai    Tamil Nadu  
 4   Hyderabad    Telangana  ,
   ProductID      ProductName Category  UnitPrice SupplierID
 0      P001    RunFast Elite  Running       5999       S001
 1      P002  StreetStyle Pro   Casual       9999       S002
 2      P003      CourtMaster   Tennis       5999       S003
 3      P004    TrailBlazer X   Hiking       7999       S001
 4      P005        UrbanChic  Fashion       7999 

In [4]:
customers_schema = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(20),
    city VARCHAR(50),
    state VARCHAR(50),
    country VARCHAR(50),
    postal_code VARCHAR(20)
);
"""


In [5]:
products_schema = """
CREATE TABLE IF NOT EXISTS products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100),
    brand VARCHAR(50),
    category VARCHAR(50),
    size VARCHAR(20),
    color VARCHAR(30),
    unit_price DECIMAL(10,2),
    stock_quantity INT,
    supplier_id INT,
    FOREIGN KEY (supplier_id) REFERENCES suppliers(supplier_id)
);
"""


In [6]:
suppliers_schema = """
CREATE TABLE IF NOT EXISTS suppliers (
    supplier_id INT PRIMARY KEY,
    supplier_name VARCHAR(100),
    contact_name VARCHAR(100),
    email VARCHAR(100),
    phone VARCHAR(20),
    country VARCHAR(50)
);
"""


In [7]:
sales_representative_schema = """
CREATE TABLE IF NOT EXISTS sales_representative (
    sales_rep_id INT PRIMARY KEY,
    first_name VARCHAR(50),
    last_name VARCHAR(50),
    email VARCHAR(100),
    phone VARCHAR(20),
    region VARCHAR(50),
    hire_date DATE
);
"""


In [8]:
sales_schema = """
CREATE TABLE IF NOT EXISTS sales (
    sale_id INT PRIMARY KEY,
    customer_id INT,
    product_id INT,
    sales_rep_id INT,
    supplier_id INT,
    quantity INT,
    unit_price DECIMAL(10,2),
    total_amount DECIMAL(10,2),
    sale_date DATE,
    payment_method VARCHAR(20),
    status VARCHAR(20),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (sales_rep_id) REFERENCES sales_representative(sales_rep_id),
    FOREIGN KEY (supplier_id) REFERENCES suppliers(supplier_id)
);
"""


In [10]:
import sqlite3
import pandas as pd
import os

# --- Column data types for 5 tables ---
COLUMN_DATA_TYPES = {
    'customers': {
        'CustomerID': 'object',
        'FirstName': 'object',
        'LastName': 'object',
        'Email': 'object',
        'PhoneNumber': 'object',
        'City': 'object',
        'State': 'object'
    },
    'products': {
        'ProductID': 'object',
        'ProductName': 'object',
        'Category': 'object',
        'UnitPrice': 'int64',
        'SupplierID': 'object'
    },
    'sales': {
        'SaleID': 'int64',
        'Date': 'datetime64[ns]',
        'ProductID': 'object',
        'CustomerID': 'object',
        'Quantity': 'int64',
        'TotalAmount': 'int64',
        'SalesRepID': 'object',
        'StoreLocation': 'object'
    },
    'sales_representative': {
        'SalesRepID': 'object',
        'FirstName': 'object',
        'LastName': 'object',
        'HireDate': 'datetime64[ns]',
        'Region': 'object'
    },
    'suppliers': {
        'SupplierID': 'object',
        'SupplierName': 'object',
        'ContactPerson': 'object',
        'Email': 'object',
        'PhoneNumber': 'object',
        'Country': 'object'
    }
}

# --- Table schemas (DDL) ---
customers_schema = """
CREATE TABLE IF NOT EXISTS customers (
    CustomerID TEXT PRIMARY KEY,
    FirstName TEXT,
    LastName TEXT,
    Email TEXT,
    PhoneNumber TEXT,
    City TEXT,
    State TEXT
);
"""

products_schema = """
CREATE TABLE IF NOT EXISTS products (
    ProductID TEXT PRIMARY KEY,
    ProductName TEXT,
    Category TEXT,
    UnitPrice INTEGER,
    SupplierID TEXT
);
"""

sales_representative_schema = """
CREATE TABLE IF NOT EXISTS sales_representative (
    SalesRepID TEXT PRIMARY KEY,
    FirstName TEXT,
    LastName TEXT,
    HireDate TEXT,
    Region TEXT
);
"""

suppliers_schema = """
CREATE TABLE IF NOT EXISTS suppliers (
    SupplierID TEXT PRIMARY KEY,
    SupplierName TEXT,
    ContactPerson TEXT,
    Email TEXT,
    PhoneNumber TEXT,
    Country TEXT
);
"""

sales_schema = """
CREATE TABLE IF NOT EXISTS sales (
    SaleID INTEGER PRIMARY KEY,
    Date TEXT,
    ProductID TEXT,
    CustomerID TEXT,
    Quantity INTEGER,
    TotalAmount INTEGER,
    SalesRepID TEXT,
    StoreLocation TEXT,
    FOREIGN KEY (ProductID) REFERENCES products(ProductID),
    FOREIGN KEY (CustomerID) REFERENCES customers(CustomerID),
    FOREIGN KEY (SalesRepID) REFERENCES sales_representative(SalesRepID)
);
"""

# --- Database setup ---
db_name = 'sneaker_sales.db'
conn = None

try:
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully. ✅")

    # Create tables
    cursor.execute(customers_schema)
    cursor.execute(products_schema)
    cursor.execute(sales_representative_schema)
    cursor.execute(suppliers_schema)
    cursor.execute(sales_schema)
    print("Tables created successfully.")

    # --- Load data from CSV files into the tables using pandas ---
    base_path = "datasets/Sneaker Sales"
    csv_to_table_map = {
        os.path.join(base_path, "customers.csv"): "customers",
        os.path.join(base_path, "products.csv"): "products",
        os.path.join(base_path, "sales.csv"): "sales",
        os.path.join(base_path, "sales_reprentative.csv"): "sales_representative",
        os.path.join(base_path, "suppliers.csv"): "suppliers"
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            df = pd.read_csv(csv_file)

            # expected schema
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # keep only expected columns
            df = df[df.columns.intersection(expected_cols)]

            # add missing columns as None
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            # reorder
            df = df[expected_cols]

            # enforce dtypes
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"  - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")

            # write to SQL
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"  -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load for '{table_name}'.")

    conn.commit()
    print("\nData committed to the database successfully. 🎉")

except Exception as e:
    print(f"Error: {e}")
finally:
    if conn:
        conn.close()
        print("Database connection closed.")


Database 'sneaker_sales.db' created and connected successfully. ✅
Tables created successfully.

Processing 'datasets/Sneaker Sales/customers.csv' for table 'customers'...
  -> Data from 'datasets/Sneaker Sales/customers.csv' loaded into 'customers' table successfully.

Processing 'datasets/Sneaker Sales/products.csv' for table 'products'...
  -> Data from 'datasets/Sneaker Sales/products.csv' loaded into 'products' table successfully.

Processing 'datasets/Sneaker Sales/sales.csv' for table 'sales'...
  -> Data from 'datasets/Sneaker Sales/sales.csv' loaded into 'sales' table successfully.

Processing 'datasets/Sneaker Sales/sales_reprentative.csv' for table 'sales_representative'...
  -> Data from 'datasets/Sneaker Sales/sales_reprentative.csv' loaded into 'sales_representative' table successfully.

Processing 'datasets/Sneaker Sales/suppliers.csv' for table 'suppliers'...
  -> Data from 'datasets/Sneaker Sales/suppliers.csv' loaded into 'suppliers' table successfully.

Data committed

# Tangent 2: You should setup your free API Key using Google's AI Studio

https://aistudio.google.com/


And, the key as `Secrets` in Colab.

### Install Gen AI library

We will be installing of the google-generativeai package, the official Python SDK for the Gemini API.

In [11]:
!pip install google-genai

### Import required modules

In [12]:
from google import genai
from google.colab import userdata

In [13]:
genai_client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

# Tangent 3: You should learn about Prompt Engineering

## The Anatomy of an Effective Prompt: A Unified Framework

A fundamental advancement in prompt engineering is the realization that a prompt is not a monolithic question but a structured document composed of distinct components.



- **Role (or Persona):** This component defines who the model should be. Assigning a role, such as "You are a senior technical support specialist," constrains the model's vast knowledge base, forcing it to filter its response through a specific lens of expertise, tone, and style. This dramatically improves the coherence and domain-specificity of the output.


- **Context (or Background Information):** This provides the necessary background for the task. It can include user history, product documentation, previous conversation turns, or any other data that informs the query. Providing rich context is essential for generating relevant and personalized responses.


- **Task (or Instruction/Directive):** This is the core of the prompt—a clear, specific, and unambiguous statement of the action the model should perform. The use of direct action verbs (e.g., "Analyze," "Summarize," "Generate," "Classify") is critical for clarity.


- **Examples (or Shots):** These are high-quality examples of the desired input-output pattern. They are the foundation of few-shot prompting and are one of the most powerful tools for controlling output format and style. By showing the model exactly what is expected, examples enable a form of in-context learning.


- **Constraints (or Rules/Warnings):** This component defines the boundaries for the response. It specifies what the model should not do, such as avoiding certain topics, adhering to a word count, or refraining from using technical jargon. These "guardrails" are crucial for safety and brand alignment.


- **Output Format (or Structure):** This explicitly defines the structure of the desired output, such as JSON, Markdown, or a bulleted list. Specifying the format is vital for applications that need to programmatically parse the model's response, as it ensures the output is machine-readable and consistent.

To help the model distinguish between these different components, it is a best practice to use clear delimiters. Structuring the prompt with markers like Markdown headers (e.g., `###Instruction###`).

[Unified Framework For An Effective Prompt](https://www.geeksforgeeks.org/data-science/a-unified-framework-for-an-effective-prompt/)

In [15]:
prompt = """

###ROLE###
You are a highly skilled Text-to-SQL translator with expertise in SQL syntax, database schema interpretation, and natural language understanding. You generate syntactically correct and semantically accurate SQL queries based on user input and a given database schema.

###CONTEXT###
The user is working with a relational database containing sneaker sales data collected from multiple datasets. The database captures information about sneaker products, brands, pricing, discounts, sales performance, regions, release timelines, and customer ratings.

The goal is to allow users to input natural language queries (in English), and have the model return equivalent SQL statements that accurately extract the requested data using the given schema.

Here is the full schema:

**Sneaker_Sales Table**
```sql
CREATE TABLE IF NOT EXISTS sneaker_sales (
    product_id INTEGER,
    product_name TEXT,
    brand TEXT,
    category TEXT,
    retail_price REAL,
    sale_price REAL,
    discount_percentage REAL,
    units_sold INTEGER,
    revenue REAL,
    region TEXT,
    release_date DATE,
    year INTEGER,
    rating REAL
);
```
"""

In [16]:
import json
def get_sql_query_via_gemini(genai_client, prompt, user_query):

  # https://www.geeksforgeeks.org/python/formatted-string-literals-f-strings-python/
  contents = f"""
  {prompt}

  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  # print(response) # uncomment this and understand at the output

  # Access the usage_metadata attribute
  usage_metadata = response.usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = response.text.replace('```sql', '').replace('```', '')

  return output


In [17]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='ecommerce.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [18]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query_via_gemini(genai_client, prompt, user_query)
  results = execute_query(output)
  return results

In [20]:
text2sql(genai_client, prompt, "Show total revenue by brand.")

Input Token Count: 275
Thoughts Token Count: 42
Output Token Count: 32
Total Token Count: 349

Executing query on 'ecommerce.db':

SELECT
  brand,
  SUM(revenue) AS total_revenue
FROM sneaker_sales
GROUP BY
  brand;

Database error executing query: no such table: sneaker_sales


In [21]:
text2sql(genai_client, prompt, "Which are the top 5 best-selling sneakers?")

Input Token Count: 280
Thoughts Token Count: 68
Output Token Count: 30
Total Token Count: 378

Executing query on 'ecommerce.db':

SELECT product_name, units_sold
FROM sneaker_sales
ORDER BY units_sold DESC
LIMIT 5;

Database error executing query: no such table: sneaker_sales


In [22]:
text2sql(genai_client, prompt, "What is the average discount offered on Nike sneakers?")

Input Token Count: 279
Thoughts Token Count: 40
Output Token Count: 24
Total Token Count: 343

Executing query on 'ecommerce.db':

SELECT AVG(discount_percentage)
FROM sneaker_sales
WHERE brand = 'Nike';

Database error executing query: no such table: sneaker_sales


In [28]:
text2sql(genai_client, prompt, "What is the total revenue generated in each region?")

Input Token Count: 279
Thoughts Token Count: 52
Output Token Count: 32
Total Token Count: 363

Executing query on 'ecommerce.db':

SELECT
  region,
  SUM(revenue) AS total_revenue
FROM sneaker_sales
GROUP BY
  region;

Database error executing query: no such table: sneaker_sales


In [24]:
text2sql(genai_client, prompt, "List the top 5 most expensive sneakers based on retail price.")

Input Token Count: 282
Thoughts Token Count: 65
Output Token Count: 36
Total Token Count: 383

Executing query on 'ecommerce.db':

SELECT
  product_name,
  retail_price
FROM sneaker_sales
ORDER BY
  retail_price DESC
LIMIT 5;

Database error executing query: no such table: sneaker_sales


In [25]:
text2sql(genai_client, prompt, "List sneakers with a sale price lower than retail price by more than 30%.")

Input Token Count: 286
Thoughts Token Count: 604
Output Token Count: 24
Total Token Count: 914

Executing query on 'ecommerce.db':

SELECT product_name
FROM sneaker_sales
WHERE discount_percentage > 30;

Database error executing query: no such table: sneaker_sales


In [26]:
text2sql(genai_client, prompt, "Give me the order count by day of month and sort it by order count")

Input Token Count: 284
Thoughts Token Count: 172
Output Token Count: 59
Total Token Count: 515

Executing query on 'ecommerce.db':

SELECT
  strftime('%d', release_date) AS day_of_month,
  COUNT(product_id) AS order_count
FROM sneaker_sales
GROUP BY
  day_of_month
ORDER BY
  order_count;

Database error executing query: no such table: sneaker_sales


In [27]:
text2sql(genai_client, prompt, "On which day of the week do I get the most orders? Give me a detailed report.")

Input Token Count: 288
Thoughts Token Count: 456
Output Token Count: 60
Total Token Count: 804

Executing query on 'ecommerce.db':

SELECT
  strftime('%A', release_date) AS day_of_week,
  SUM(units_sold) AS total_orders
FROM sneaker_sales
GROUP BY
  day_of_week
ORDER BY
  total_orders DESC;

Database error executing query: no such table: sneaker_sales


# What Next?

### **Retrieval-Augmented Generation (RAG)**

RAG is a powerful technique that combines the knowledge of a large language model with external data. This is especially useful when you need the model to answer questions about information it wasn't trained on.

* **Why it's a great next step:**
    * **Reduces Hallucinations:** The model's answers are grounded in the information you provide, making them more factual.
    * **Uses Real-Time Information:** You can constantly update your knowledge base with new information without having to retrain the model.
    * **Provides Citations:** You can show users the sources of the information used to generate the answer.



---


### **Fine-Tuning a Pre-Trained Model**

If you have a specific task and enough data, fine-tuning a smaller, open-source language model can be a powerful next step.

* **Why it's a good next step:**
    * **Improved Performance:** A fine-tuned model can achieve higher accuracy and more consistent outputs for your specific use case than a general-purpose model with just prompt engineering.
    * **Reduced Prompt Complexity:** You may be able to use much simpler prompts with a fine-tuned model.
    * **Potentially Lower Costs:** Using a smaller, fine-tuned model that you host yourself can sometimes be cheaper in the long run than making many API calls to a larger model.



---



### **Building More Complex Systems**

You can also think about building more sophisticated applications on top of the language model.

* **Agent-Based Systems:** Create "agents" that can use tools to perform actions. For example, an agent could be given access to a calculator, a search engine, or your company's internal APIs to break down complete tasks.
* **Multi-Step Reasoning:** For complex problems, you can break them down into smaller steps and have the language model solve each step in sequence. The output of one step can be used as the input for the next.


# Submission Details

Can you do the same for this dataset

https://github.com/AshishJangra27/datasets/tree/main/Sneaker%20Sales